In [0]:
# creatr 8 tables dim_material

# create 8 tables 
dim_material
dim_supply
fact_demand
fact_supply
fact_inventory
fact_transport
fact_warehouse

In [0]:
#                        SILVER
#                          │
#        ┌─────────────────┼──────────────────┐
#        │                 │                  │
#        ▼                 ▼                  ▼
#  dim_material        dim_plant        dim_supplier
#        │
#        ├─────────────────────────────────────┐
#        │                                     │
#       ▼                                     ▼
#  fact_demand                           fact_supply
#        │                                     │
#        ├───────────────┐                     │
#        ▼               ▼                     ▼
# fact_inventory   fact_warehouse       fact_transport 

In [0]:
# Create dim_material
#  Here we are trimming spaces ,standardizing names , converting numeric fields ,converting date ,removing duplicate material IDs

from pyspark.sql import functions as F

# Read Bronze MARA
mara = spark.table("workspace.bronze.mara")

# Transform into Silver material dimension
dim_material = (
    mara
    .select(
        F.trim(F.col("MATNR")).alias("material_id"),
        F.trim(F.col("MATKL")).alias("material_group"),
        F.trim(F.col("MEINS")).alias("base_uom"),
        F.col("UNIT_PRICE").cast("double").alias("unit_price"),
        F.col("BASE_DAILY_DEMAND").cast("double").alias("base_daily_demand"),
        F.to_date(F.col("ERSDA")).alias("created_date")
    )
    .filter(F.col("material_id").isNotNull())
    .dropDuplicates(["material_id"])
)

display(dim_material)

In [0]:
# write to silver
dim_material.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.dim_material")

In [0]:
%sql
SELECT *
FROM workspace.silver.dim_material
LIMIT 10;

In [0]:
from pyspark.sql import functions as F

# ============================================================
# 1. READ BRONZE TABLES
# ============================================================

mara = spark.table("workspace.bronze.mara")
marc = spark.table("workspace.bronze.marc")
mard = spark.table("workspace.bronze.mard")
t001w = spark.table("workspace.bronze.t001w")
lfa1 = spark.table("workspace.bronze.lfa1")

vbak = spark.table("workspace.bronze.vbak")
vbap = spark.table("workspace.bronze.vbap")

ekko = spark.table("workspace.bronze.ekko")
ekpo = spark.table("workspace.bronze.ekpo")
eket = spark.table("workspace.bronze.eket")

shipment_header = spark.table("workspace.bronze.shipment_header")
tracking_event = spark.table("workspace.bronze.tracking_event")
carrier_master = spark.table("workspace.bronze.carrier_master")

wms_inventory = spark.table("workspace.bronze.wms_inventory")
pick_task = spark.table("workspace.bronze.pick_task")

# ============================================================
# 3. DIM PLANT
# ============================================================

dim_plant = (
    t001w
    .select(
        F.trim("WERKS").alias("plant_id"),
        *[
            F.col(c).alias(c.lower())
            for c in t001w.columns
            if c != "WERKS"
        ]
    )
    .filter(F.col("plant_id").isNotNull())
    .dropDuplicates(["plant_id"])
)

dim_plant.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.dim_plant")


# ============================================================
# 4. DIM SUPPLIER
# ============================================================

dim_supplier = (
    lfa1
    .select(
        F.trim("LIFNR").alias("supplier_id"),
        *[
            F.col(c).alias(c.lower())
            for c in lfa1.columns
            if c != "LIFNR"
        ]
    )
    .filter(F.col("supplier_id").isNotNull())
    .dropDuplicates(["supplier_id"])
)

dim_supplier.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.dim_supplier")


# ============================================================
# 5. FACT DEMAND
#    VBAK + VBAP
# ============================================================

fact_demand = (
    vbap.alias("item")
    .join(
        vbak.alias("header"),
        F.col("item.VBELN") == F.col("header.VBELN"),
        "left"
    )
    .select(
        F.col("item.VBELN").alias("sales_order_id"),
        F.col("item.POSNR").alias("sales_order_item"),
        F.col("item.MATNR").alias("material_id"),
        *[
            F.col(f"item.{c}").alias(f"item_{c.lower()}")
            for c in vbap.columns
            if c not in ["VBELN", "POSNR", "MATNR"]
        ],
        *[
            F.col(f"header.{c}").alias(f"header_{c.lower()}")
            for c in vbak.columns
            if c != "VBELN"
        ]
    )
)

fact_demand.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.fact_demand")


# ============================================================
# 6. FACT SUPPLY
#    EKKO + EKPO + EKET
# ============================================================

fact_supply = (
    ekpo.alias("item")
    .join(
        ekko.alias("header"),
        F.col("item.EBELN") == F.col("header.EBELN"),
        "left"
    )
    .join(
        eket.alias("schedule"),
        (
            (F.col("item.EBELN") == F.col("schedule.EBELN")) &
            (F.col("item.EBELP") == F.col("schedule.EBELP"))
        ),
        "left"
    )
    .select(
        F.col("item.EBELN").alias("po_id"),
        F.col("item.EBELP").alias("po_item"),
        F.col("item.MATNR").alias("material_id"),
        F.col("item.WERKS").alias("plant_id")
        if "WERKS" in ekpo.columns
        else F.lit(None).cast("string").alias("plant_id"),

        *[
            F.col(f"item.{c}").alias(f"item_{c.lower()}")
            for c in ekpo.columns
            if c not in ["EBELN", "EBELP", "MATNR", "WERKS"]
        ],

        *[
            F.col(f"header.{c}").alias(f"header_{c.lower()}")
            for c in ekko.columns
            if c != "EBELN"
        ],

        *[
            F.col(f"schedule.{c}").alias(f"schedule_{c.lower()}")
            for c in eket.columns
            if c not in ["EBELN", "EBELP"]
        ]
    )
)

fact_supply.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.fact_supply")


# ============================================================
# 7. FACT INVENTORY
#    MARD + MARC
# ============================================================

fact_inventory = (
    mard.alias("inv")
    .join(
        marc.alias("plan"),
        (
            (F.col("inv.MATNR") == F.col("plan.MATNR")) &
            (F.col("inv.WERKS") == F.col("plan.WERKS"))
        ),
        "left"
    )
    .select(
        F.col("inv.MATNR").alias("material_id"),
        F.col("inv.WERKS").alias("plant_id"),

        *[
            F.col(f"inv.{c}").alias(f"inventory_{c.lower()}")
            for c in mard.columns
            if c not in ["MATNR", "WERKS"]
        ],

        *[
            F.col(f"plan.{c}").alias(f"planning_{c.lower()}")
            for c in marc.columns
            if c not in ["MATNR", "WERKS"]
        ]
    )
)

fact_inventory.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.fact_inventory")


# ============================================================
# 8. FACT TRANSPORT
#    Shipment + Tracking + Carrier
# ============================================================

fact_transport = (
    shipment_header.alias("s")
    .join(
        tracking_event.alias("t"),
        F.col("s.shipment_id") == F.col("t.shipment_id"),
        "left"
    )
    .join(
        carrier_master.alias("c"),
        F.col("s.carrier_id") == F.col("c.carrier_id"),
        "left"
    )
    .select(
        *[
            F.col(f"s.{c}").alias(f"shipment_{c.lower()}")
            for c in shipment_header.columns
        ],
        *[
            F.col(f"t.{c}").alias(f"tracking_{c.lower()}")
            for c in tracking_event.columns
            if c != "shipment_id"
        ],
        *[
            F.col(f"c.{c}").alias(f"carrier_{c.lower()}")
            for c in carrier_master.columns
            if c != "carrier_id"
        ]
    )
)

fact_transport.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.fact_transport")


# ============================================================
# 9. FACT WAREHOUSE
#    WMS Inventory + Pick Task
# ============================================================

fact_warehouse = (
    wms_inventory.alias("w")
    .join(
        pick_task.alias("p"),
        (
            (F.col("w.material_id") == F.col("p.material_id")) &
            (F.col("w.warehouse_id") == F.col("p.warehouse_id"))
        ),
        "left"
    )
    .select(
        *[
            F.col(f"w.{c}").alias(f"inventory_{c.lower()}")
            for c in wms_inventory.columns
        ],
        *[
            F.col(f"p.{c}").alias(f"pick_{c.lower()}")
            for c in pick_task.columns
            if c not in ["material_id", "warehouse_id"]
        ]
    )
)

fact_warehouse.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.fact_warehouse")


# ============================================================
# 10. FINAL MESSAGE
# ============================================================

print("==============================================")
print("SILVER TRANSFORMATION COMPLETED")
print("==============================================")

print("Created:")
print("1. workspace.silver.dim_material")
print("2. workspace.silver.dim_plant")
print("3. workspace.silver.dim_supplier")
print("4. workspace.silver.fact_demand")
print("5. workspace.silver.fact_supply")
print("6. workspace.silver.fact_inventory")
print("7. workspace.silver.fact_transport")
print("8. workspace.silver.fact_warehouse")

print("==============================================")

In [0]:
%sql
-- validate all silver tables
SELECT 'dim_material' AS table_name, COUNT(*) AS row_count
FROM workspace.silver.dim_material

UNION ALL
SELECT 'dim_plant', COUNT(*)
FROM workspace.silver.dim_plant

UNION ALL
SELECT 'dim_supplier', COUNT(*)
FROM workspace.silver.dim_supplier

UNION ALL
SELECT 'fact_demand', COUNT(*)
FROM workspace.silver.fact_demand

UNION ALL
SELECT 'fact_supply', COUNT(*)
FROM workspace.silver.fact_supply

UNION ALL
SELECT 'fact_inventory', COUNT(*)
FROM workspace.silver.fact_inventory

UNION ALL
SELECT 'fact_transport', COUNT(*)
FROM workspace.silver.fact_transport

UNION ALL
SELECT 'fact_warehouse', COUNT(*)
FROM workspace.silver.fact_warehouse

ORDER BY table_name;

In [0]:
%sql
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT material_id) AS unique_materials
FROM workspace.silver.dim_material;

In [0]:
%sql
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT plant_id) AS unique_plants
FROM workspace.silver.dim_plant;

In [0]:
%sql
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT CONCAT(material_id, '|', plant_id)) AS material_plant_combinations
FROM workspace.silver.fact_inventory;